# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and example for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and adheres to the [MLCommons Croissant specification](https://mlcommons.org/croissant/).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q 'mlcroissant>=0.8.4' matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant` and preview the main dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets and fields, and inspect their `@id` values for subsequent referencing. Croissant schemas describe tabular datasets in terms of *record sets* (like tables) and *fields* (columns), each uniquely identified by their `@id`.


In [ ]:
# Display all record sets, their @id, and fields

record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f'Record Set Name: {rs.name}  (@id: {rs.id})')
    print('  Description:', getattr(rs, 'description', 'N/A'))
    print('  Fields:')
    for f in rs.fields:
        print(f"    - {f.name}  (@id: {f.id}) type: {f.data_type}")
    print('---')

### Print a few records from the first record set
This gives a sample of the actual structured data as parsed by Croissant. Adjust the `record_set_id` variable as desired.

In [ ]:
# Let's inspect records from the first (or main) record set

# Select the main record set (update if needed)
main_record_set = record_sets[0]  # or find by name
main_record_set_id = main_record_set.id

print(f"Sample records from record set: {main_record_set.name} (@id: {main_record_set_id})\n")
# Print first 2 records as sample
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i >= 1:
        break

## 3. Data Extraction
Load all data from each record set into pandas DataFrames for further analysis. Use the record set and field `@id`s as referenced above.

In [ ]:
# Extract all record sets into DataFrames

dataframes = {}
for rs in record_sets:
    print(f"Loading records for record set: {rs.name} (@id: {rs.id})")
    try:
        df = pd.DataFrame(dataset.records(record_set=rs.id))
        dataframes[rs.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
        print('---')
    except Exception as e:
        print(f"  Could not load: {e}")
        print('---')

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate data processing steps for one main record set, including filtering by a numeric field, normalization, and grouping by a categorical field.

You can update the selected field `@id` values as needed. We'll infer likely numeric and group fields from the field types.

In [ ]:
import numpy as np

# Choose main record set (update if needed)
rs = main_record_set
df = dataframes[rs.id]

# List all numeric fields and pick one with type 'Number', 'Float', or 'Integer'
numeric_fields = [f for f in rs.fields if f.data_type in ('Number', 'Float', 'Integer')]
print('Numeric fields available:')
for nf in numeric_fields:
    print(f"  {nf.name} (@id: {nf.id})")
if not numeric_fields:
    raise Exception('No numeric field found.')

# Try to use the first numeric field
numeric_field_id = numeric_fields[0].id

# Find a grouping field (preferably a categorical or string type)
group_fields = [f for f in rs.fields if f.data_type in ('Text', 'String')]
group_field_id = group_fields[0].id if group_fields else None

print(f"\nUsing numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Using grouping field: {group_field_id}\n")
else:
    print(f"No clear categorical field for grouping.\n")

# Drop rows with missing values for numeric field
df_clean = df.copy()
df_clean = df_clean[df_clean[numeric_field_id].notnull()]
df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')
df_clean = df_clean[df_clean[numeric_field_id].notnull()]

# Define a threshold at median value for demonstration (you may choose another value)
threshold = df_clean[numeric_field_id].median()
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows")
print(filtered_df.head(2))

# Normalize numeric field (z-score)
filtered_df[f'{numeric_field_id}_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Group by group_field_id and aggregate (mean)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head(3))

## 5. Visualization
Let's visualize the distribution of our selected numeric field and its relationship with a grouping field, if available.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of numeric field
plt.figure(figsize=(7,4))
plt.hist(df_clean[numeric_field_id], bins=30, color='cornflowerblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field is available, show mean per group
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,4))
    grouped_df.set_index(group_field_id)[numeric_field_id].plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry protein dataset using the `mlcroissant` library. We loaded record sets and fields by their `@id`, extracted data into DataFrames, and performed basic EDA and visualization. For deeper analysis, consult the [dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and Croissant schema.